<a href="https://colab.research.google.com/github/jayx003/GenAI-Practice/blob/main/06_sampling_controls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sampling Controls

Control the creativity and randomness of LLM outputs.

**Temperature**, **Top-p**, **Top-k**,  — the knobs that shape how the model generates text.

https://chatgpt.com/share/6a0b2ba1-a214-83a3-a85e-7437ebc8cdb6

## Setup

In [1]:
from google import genai
from google.genai import types
import os
# from dotenv import load_dotenv

# load_dotenv()
from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')
# API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)

## Temperature

Controls randomness. Higher = more creative/random. Lower = more focused/deterministic.

- **0.0** — Deterministic, always picks the most likely token
- **0.5** — Balanced
- **1.0** — Default, good balance
- **2.0** — Very creative, may be incoherent

In [2]:
def generate(prompt, temperature=0.0):
    """Generate text with specified temperature."""
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=temperature
        )
    )
    return response.text

prompt = "Write a one-sentence story about a robot."

print("=== Temperature 0.0 (Deterministic) ===")
for i in range(2):
    print(f"{i+1}. {generate(prompt, temperature=0.0)}")

=== Temperature 0.0 (Deterministic) ===
1. Designed to wait for a creator who never returned, the rusty automaton spent centuries meticulously cataloging the falling stars.
2. Designed to wait for a creator who never returned, the rusty automaton spent centuries tending a garden that had long since turned to dust.


In [3]:
# Compare temperatures
prompt = "Write a one-sentence story about a robot."

for temp in [0.5, 1.0, 1.5,2.0]:
    print(f"\n=== Temperature {temp} ===")
    print(generate(prompt, temperature=temp))


=== Temperature 0.5 ===
After centuries of tending to the empty planet, the lonely little robot finally picked up a faint, answering signal from the stars.

=== Temperature 1.0 ===
After two centuries of solitude on a dead Earth, the rusty caretaker finally heard the distant, unmistakable crunch of a tiny, living footstep.

=== Temperature 1.5 ===
Designed to mine barren asteroids, the lonely automaton spent its centuries whispering poetry into the void just to hear the echo of its own vocal processors.

=== Temperature 2.0 ===
Designed to heal a dying planet, the lonely rover wept rusted tears upon finding the first blade of green grass.


## Top-p (Nucleus Sampling)

Keep adding tokens until their **cumulative probability** reaches p.

**Example:** Tokens ranked by probability:
```
Token A: 40% → cumulative: 40%
Token B: 30% → cumulative: 70%
Token C: 15% → cumulative: 85%
Token D: 10% → cumulative: 95%
Token E:  5% → cumulative: 100%
```

- **Top-p 0.5** → Only ~2 tokens (A, partial B) → **Focused**
- **Top-p 0.95** → 4 tokens (A, B, C, D) → **More diverse**

**Lower p = fewer tokens = more focused**

In [10]:
def generate_topp(prompt, top_p=1.0):
    """Generate text with specified top_p."""
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(
            top_p=top_p,
            temperature=1.0  # Keep temperature constant
        )
    )
    return response.text

prompt = "List 5 creative usage of AI"

print("=== Top-p 0.3 (Focused) ===")
print(generate_topp(prompt, top_p=0.3))

print("\n=== Top-p 0.95 (More Creative) ===")
print(generate_topp(prompt, top_p=0.95))

=== Top-p 0.3 (Focused) ===
Here are 5 creative and unconventional ways artificial intelligence is being used today, moving beyond standard text generation and chatbots:

### 1. Generating Micro-Interiors for Miniature Photography
* **The Concept:** Miniature photography (using tiny figures and dioramas) requires building physical, highly detailed backdrops, which takes days or weeks. 
* **AI Application:** Creators are using AI image generators (like Midjourney or Stable Diffusion) to create hyper-realistic backgrounds, landscapes, and interiors tailored to the exact perspective of their tiny subjects. They then display the AI image on a high-res monitor behind the miniature set, instantly blending physical photography with limitless virtual worlds.

### 2. Interactive "Memory Museums" for Family History
* **The Concept:** Preserving the stories of aging grandparents or ancestors usually results in static photo albums or standard video interviews.
* **AI Application:** Families are fe

## Top-k

Only consider the **top k most likely tokens** (by count, not probability).

**Example:** Tokens ranked by probability:
```
Token 1: 35%  ← Top 1
Token 2: 25%  ← Top 2
Token 3: 15%  ← Top 3
Token 4: 10%  ← Top 4
Token 5:  8%  ← Top 5
Token 6:  4%
Token 7:  2%
...
```

- **Top-k 1** → Only Token 1 → **Greedy (deterministic)**
- **Top-k 3** → Tokens 1, 2, 3 → **Focused**
- **Top-k 40** → Top 40 tokens → **Balanced (default)**
- **Top-k 100** → Top 100 tokens → **More diverse**

**Lower k = fewer tokens = more focused**

**Top-k vs Top-p:**
- Top-k: Fixed number of tokens
- Top-p: Variable number based on cumulative probability

In [7]:
def generate_topk(prompt, top_k=40):
    """Generate text with specified top_k."""
    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(
            top_k=top_k,
            temperature=1.0
        )
    )
    return response.text

prompt = "Invent a new word and define it:"

print("=== Top-k 1 (Greedy) ===")
print(generate_topk(prompt, top_k=1))

print("\n=== Top-k 40 (Default) ===")
print(generate_topk(prompt, top_k=40))

=== Top-k 1 (Greedy) ===
**Word:** 
**Chronosnack** (noun)

**Pronunciation:** /KRON-oh-snak/

**Definition:** 
1. A small, seemingly insignificant food item (like a single handful of chips, a cube of cheese, or a cookie) consumed while waiting for a larger meal to cook, which accidentally ruins your appetite. 
2. The act of grazing on leftovers in the kitchen for precisely two minutes while staring blankly into the open refrigerator.

**Etymology:** 
Derived from *chronos* (Greek for time) and *snack*. 

**Example in a sentence:** 
*"I didn't mean to ruin dinner, but I fell victim to a devastating **chronosnack** right as the pasta was boiling."*

=== Top-k 40 (Default) ===
**Word:** *Somnamblot* (noun)

**Pronunciation:** /sɒmˈnæm.blɒt/

**Definition:** 
1. The blurry, half-conscious state of mind experienced in the first few minutes after waking up, during which a person is physically moving or interacting with the world, but their brain is still operating entirely in "dream logic."

## Combining Parameters

In [13]:
def generate_custom(prompt, temperature=1.0, top_p=0.95, top_k=40, seed=None):
    """Generate with all sampling controls."""
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    if seed is not None:
        config.seed = seed

    response = client.models.generate_content(
        model='gemini-3.5-flash-lite',
        contents=prompt,
        config=config
    )
    return response.text

prompt = "Write a haiku about AI:"

# Focused, deterministic
print("=== Focused (temp=0.3, top_p=0.5, top_k=10) ===")
print(generate_custom(prompt, temperature=0.3, top_p=0.5, top_k=10))

# Creative, diverse
print("\n=== Creative (temp=1.5, top_p=0.95, top_k=100) ===")
print(generate_custom(prompt, temperature=1.5, top_p=0.95, top_k=40))

=== Focused (temp=0.3, top_p=0.5, top_k=10) ===
Silicon and code,
Thinking in a sea of stars,
New mind wakes at dawn.

=== Creative (temp=1.5, top_p=0.95, top_k=100) ===
Silent circuits think,
Growing smarter every day,
Worlds inside the screen.


## Use Case Presets

In [14]:
# Define presets for different use cases
PRESETS = {
    "factual": {"temperature": 0.0, "top_p": 0.5, "top_k": 10},
    "balanced": {"temperature": 0.7, "top_p": 0.9, "top_k": 40},
    "creative": {"temperature": 1.2, "top_p": 0.95, "top_k": 100},
    "brainstorm": {"temperature": 1.5, "top_p": 0.98, "top_k": 150},
}

prompt = "What are some ways to improve productivity?"

for name, params in PRESETS.items():
    print(f"\n=== {name.upper()} ===")
    print(generate_custom(prompt, **params))


=== FACTUAL ===
Improving productivity isn’t about working *more* hours; it’s about working *smarter* so you can achieve better results with less stress. 

Here are some of the most effective, science-backed ways to improve productivity, broken down into key categories:

---

### 1. Optimize Your Time and Focus
*   **The Pomodoro Technique:** Work with intense focus for 25 minutes, then take a 5-minute break. After four cycles, take a longer break (15–30 minutes). This prevents mental fatigue and keeps motivation high.
*   **Time Blocking:** Instead of a traditional to-do list, schedule specific blocks of time on your calendar for specific tasks (e.g., 9:00 AM – 10:30 AM: Deep Work; 11:00 AM – 11:30 AM: Email). 
*   **Eat the Frog:** Tackle your hardest, most important task first thing in the morning. Once it’s done, the rest of your day will feel easier and less stressful.
*   **The 2-Minute Rule:** If a task takes two minutes or less to do (e.g., replying to a quick text, filing a d

## Key Takeaways

| Parameter | What it does | Low value | High value |
|-----------|-------------|-----------|------------|
| **Temperature** | Controls randomness | Focused, deterministic | Creative, random |
| **Top-p** | Nucleus sampling | Only top tokens | More diverse tokens |
| **Top-k** | Limits token choices | Few choices (greedy) | Many choices |
| **Seed** | Reproducibility | Random each time | Same output |

**Common combinations:**
- **Factual/Code**: temp=0, top_p=0.5, top_k=10
- **General chat**: temp=0.7, top_p=0.9, top_k=40
- **Creative writing**: temp=1.2, top_p=0.95, top_k=100
- **Brainstorming**: temp=1.5, top_p=0.98, top_k=150

---

**Tip:** Temperature is the most important. Start there, then fine-tune with top_p and top_k.